# Phase 2.3 — Feature processing and family comparison

Inputs: `features/<family>_<SIZE>_<NORM>.npy` (from `src/extract_features.py`),
`data/master.csv`, `results/splits.csv`.
Outputs: tables in `results/tables/`, figures in `results/figures/`.

Everything is scored with **cross-validation on the training set only**, using the
frozen folds from `splits.csv`. The test set is not touched in this phase.

Change `SIZE` / `NORM` in the first cell and re-run to produce the 448 and z-score
comparisons.

In [ ]:
import json, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.decomposition import PCA
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.feature_selection import SelectFromModel, VarianceThreshold
from sklearn.metrics import (average_precision_score, fbeta_score,
                             precision_score, recall_score)
from sklearn.model_selection import PredefinedSplit, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

# ---- configuration: change these two and re-run the notebook ----
SIZE = 256
NORM = "clahe"          # "clahe" | "zscore" | "none"
SEED = 42
N_JOBS = -1             # lower to 2 if your laptop runs out of memory
# -----------------------------------------------------------------

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
FEAT, FIG, TAB = ROOT / "features", ROOT / "results" / "figures", ROOT / "results" / "tables"
FIG.mkdir(parents=True, exist_ok=True); TAB.mkdir(parents=True, exist_ok=True)

REGIONS = ["hand", "leg", "hip", "shoulder"]
sns.set_theme(style="whitegrid", context="notebook")

# ---- alignment: the .npy row order is defined by the ids file, NOT by master.csv ----
ids    = pd.read_csv(FEAT / f"images_{SIZE}_ids.csv")
master = pd.read_csv(ROOT / "data" / "master.csv")
splits = pd.read_csv(ROOT / "results" / "splits.csv")

df = (ids
      .merge(master[["image_id", "region_group", "fractured"]], on="image_id",
             how="left", validate="one_to_one")
      .merge(splits[["image_id", "split", "fold"]], on="image_id",
             how="left", validate="one_to_one"))
assert len(df) == len(ids), "merge changed the row count"
assert df.notna().all().all(), "some images are missing from master.csv or splits.csv"

y        = df["fractured"].to_numpy()
region   = df["region_group"].to_numpy()
is_train = (df["split"] == "train").to_numpy()
cv       = PredefinedSplit(df.loc[is_train, "fold"].to_numpy())   # the FROZEN folds

print(f"{len(df)} images   train {is_train.sum()}   test {(~is_train).sum()}")
print(f"fractured: all {y.sum()}   train {y[is_train].sum()}   test {y[~is_train].sum()}")
print(f"CV folds: {cv.get_n_splits()}")
print("\ntrain images per fold x fractured:")
print(pd.crosstab(df.loc[is_train, "fold"], df.loc[is_train, "fractured"]))

### Load the feature families

Each family lives in its own file so they can be combined freely. Row order is
identical across all of them because they were all written from the same cached
image array.

In [ ]:
FAMILIES = ["hog", "lbp", "glcm", "gabor", "intensity"]

F = {}
for name in FAMILIES:
    p = FEAT / f"{name}_{SIZE}_{NORM}.npy"
    if not p.exists():
        print(f"MISSING {p.name} - run: python src/extract_features.py "
              f"--size {SIZE} --norm {NORM} --families {name}")
        continue
    F[name] = np.load(p)
    assert len(F[name]) == len(df), f"{name} has {len(F[name])} rows, expected {len(df)}"

info = pd.DataFrame({
    "family": list(F),
    "dim": [F[k].shape[1] for k in F],
    "MB": [round(F[k].nbytes / 1e6, 1) for k in F],
    "constant_cols": [int((F[k].std(axis=0) == 0).sum()) for k in F],
})
print(info.to_string(index=False))
print(f"\ntotal dimension: {info['dim'].sum()}")

## 1. Feature family comparison

One fixed model (linear SVM, `class_weight="balanced"`), the frozen 5 folds, no
test data. The point is to rank feature sets, not to tune a model — tuning happens
in Phase 3.

**PR-AUC is the headline metric**, not accuracy. At 17.6% positives, a model that
predicts "no fracture" for everything scores 82% accuracy and is useless. PR-AUC is
also threshold-free, so it doesn't depend on where the decision boundary happens to
sit. F2 is reported too because it weights recall over precision, which is what
matters for a missed fracture.

`VarianceThreshold` drops the zero-variance columns (the black padded corners in
HOG) — standardising those would divide by zero.

In [ ]:
def evaluate(X, name, reducer=None, cache={}):
    """Cross-validated score for one feature matrix. Scaling and any reduction are
    fitted INSIDE each fold by the Pipeline, so nothing leaks across folds."""
    Xtr, ytr, rtr = X[is_train], y[is_train], region[is_train]

    steps = [("var", VarianceThreshold(0.0)), ("sc", StandardScaler())]
    if reducer is not None:
        steps.append(("reduce", reducer))
    steps.append(("clf", LinearSVC(C=1.0, class_weight="balanced", dual="auto",
                                   max_iter=5000, random_state=SEED)))

    t0 = time.time()
    s = cross_val_predict(Pipeline(steps), Xtr, ytr, cv=cv,
                          method="decision_function", n_jobs=N_JOBS)
    pred = (s > 0).astype(int)

    # how many features actually reach the classifier? One extra fit on the full
    # training set, for reporting only - the scores above come from the CV folds.
    kept = Pipeline(steps[:-1]).fit_transform(Xtr, ytr).shape[1]

    row = {"setup": name, "dim": X.shape[1], "kept": kept,
           "PR_AUC": average_precision_score(ytr, s),
           "recall": recall_score(ytr, pred),
           "precision": precision_score(ytr, pred, zero_division=0),
           "F2": fbeta_score(ytr, pred, beta=2, zero_division=0)}
    for g in REGIONS:
        m = rtr == g
        row[f"AP_{g}"] = (average_precision_score(ytr[m], s[m])
                          if ytr[m].sum() > 1 else np.nan)
    row["sec"] = round(time.time() - t0)
    cache[name] = s
    print(f"  {name:28s} dim {X.shape[1]:5d} -> {kept:5d}  PR-AUC {row['PR_AUC']:.3f}  "
          f"F2 {row['F2']:.3f}  ({row['sec']}s)", flush=True)
    return row


# the trivial reference point: what PR-AUC does random guessing give?
baseline = y[is_train].mean()
print(f"Baseline PR-AUC (always-positive / random ranking): {baseline:.3f}\n")

rows = [evaluate(F[k], k) for k in F]

In [ ]:
COMBOS = {
    "hog+lbp":            ["hog", "lbp"],
    "hog+lbp+glcm":       ["hog", "lbp", "glcm"],
    "all - intensity":    ["hog", "lbp", "glcm", "gabor"],
    "all families":       ["hog", "lbp", "glcm", "gabor", "intensity"],
}

for name, parts in COMBOS.items():
    if all(p in F for p in parts):
        rows.append(evaluate(np.hstack([F[p] for p in parts]), name))

comp = pd.DataFrame(rows).sort_values("PR_AUC", ascending=False).reset_index(drop=True)
comp.to_csv(TAB / f"features_family_comparison_{SIZE}_{NORM}.csv", index=False)
comp.round(3)

**The intensity ablation.** Compare `all families` against `all - intensity`. Section 9
of `data_decisions.md` warns that intensity statistics encode the region (median
intensity: hand 43, leg 44, hip 59, shoulder 90) and that fracture rates differ by
region, so a pooled model can use them as a shortcut. If adding intensity lifts the
overall score but not the per-region APs, that is the shortcut showing up — the gain
is coming from guessing the region, not from seeing the fracture.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))

d = comp.sort_values("PR_AUC")
ax[0].barh(d["setup"], d["PR_AUC"], color=sns.color_palette("colorblind")[0])
ax[0].axvline(baseline, color="k", ls="--", lw=1, label=f"baseline {baseline:.2f}")
ax[0].set_xlabel("PR-AUC (cross-validated)"); ax[0].legend()
ax[0].set_title(f"Feature sets, {SIZE}px / {NORM}")

per = comp.set_index("setup")[[f"AP_{g}" for g in REGIONS]]
per.columns = REGIONS
sns.heatmap(per, annot=True, fmt=".2f", cmap="viridis", ax=ax[1], cbar_kws={"label": "AP"})
ax[1].set_title("Average precision per region")

plt.tight_layout()
plt.savefig(FIG / f"features_family_comparison_{SIZE}_{NORM}.png", dpi=150)
plt.show()

## 2. Dimensionality reduction

8,641 features against 3,266 training images — far more dimensions than samples.
This is the curse-of-dimensionality setting the plan asks you to justify, so compare
doing nothing against three ways of cutting it down.

**Note on PCA:** `PCA(n_components=0.95)` forces a full SVD and takes minutes per
fit at this width, which inside 5-fold CV is unusable. Instead fit a randomized PCA
once to read off where 95% of variance lands, then use that fixed component count
inside the CV pipeline. Same target, seconds instead of minutes.

In [ ]:
BEST = comp.iloc[0]["setup"]
parts = COMBOS.get(BEST, [BEST])
X_best = np.hstack([F[p] for p in parts])
print(f"Best feature set from section 1: {BEST}  (dim {X_best.shape[1]})")

# how many components reach 95% of variance?
Xs = StandardScaler().fit_transform(X_best[is_train])
K_PROBE = min(400, Xs.shape[0] - 1, Xs.shape[1])
pca = PCA(n_components=K_PROBE, svd_solver="randomized", random_state=SEED).fit(Xs)
cum = np.cumsum(pca.explained_variance_ratio_)
K95 = int(np.searchsorted(cum, 0.95) + 1) if cum[-1] >= 0.95 else K_PROBE

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(range(1, len(cum) + 1), cum)
ax.axhline(0.95, color="k", ls="--", lw=1)
if cum[-1] >= 0.95:
    ax.axvline(K95, color="r", ls="--", lw=1, label=f"95% at {K95} components")
    ax.legend()
else:
    ax.set_title(f"95% not reached within {K_PROBE} components "
                 f"(reached {cum[-1]:.2f}) - using {K_PROBE}")
ax.set_xlabel("components"); ax.set_ylabel("cumulative explained variance")
plt.tight_layout(); plt.savefig(FIG / f"features_pca_variance_{SIZE}_{NORM}.png", dpi=150)
plt.show()
print(f"Using {K95} PCA components ({cum[min(K95, len(cum)) - 1]:.3f} of variance)")
del Xs

In [ ]:
REDUCERS = {
    "none": None,
    f"PCA ({K95} comps)": PCA(n_components=K95, svd_solver="randomized", random_state=SEED),
    "L1 selection": SelectFromModel(
        LinearSVC(C=0.05, penalty="l1", dual=False, class_weight="balanced",
                  max_iter=3000, random_state=SEED)),
    "tree selection": SelectFromModel(
        ExtraTreesClassifier(n_estimators=50, class_weight="balanced",
                             random_state=SEED, n_jobs=N_JOBS)),
}

red_rows = [evaluate(X_best, f"{BEST} | {label}", reducer=r)
            for label, r in REDUCERS.items()]

red = pd.DataFrame(red_rows).sort_values("PR_AUC", ascending=False).reset_index(drop=True)
red.to_csv(TAB / f"features_reduction_comparison_{SIZE}_{NORM}.csv", index=False)
red.round(3)

## 3. Findings

Write these up yourself from the two tables above. The points worth settling:

- Which single family is strongest, and which is weakest.
- Does combining families beat the best single one, and by how much.
- `all families` vs `all - intensity`: did intensity help overall, and did it help the
  per-region APs? Record the answer against section 9 of `data_decisions.md`.
- Which reduction wins, and whether the cost in training time is worth it.
- Which region is hardest, and whether that tracks its training-set size
  (hand 1,025 / leg 1,699 / hip 256 / shoulder 267 training images).
- The feature set you are carrying into Phase 3, stated explicitly.

Then re-run the notebook with `NORM = "zscore"` and with `SIZE = 448` and compare.